In [ ]:
import pickle
import pandas as pd

# load df_with_dobyr.pkl
with open('df_with_dobyr.pkl', 'rb') as f:
    df_with_dobyr = pickle.load(f)

In [ ]:
import pandas
df_with_dobyr.head()

In [ ]:
diab_phecodes = df_with_dobyr[df_with_dobyr['phenotype'].str.contains('diabetes', case=False)]['phecode']
print(set(diab_phecodes))


In [ ]:
print(set(diab_phecodes))

In [ ]:
diab_phecodes_types = df_with_dobyr[df_with_dobyr['phenotype'].str.contains('diabetes', case=False)][['phecode', 'phenotype']]

import matplotlib.pyplot as plt

# Count occurrences of each diabetes phecode/phenotype
diab_counts = diab_phecodes_types['phenotype'].value_counts().sort_values(ascending=False)

# Plot bar chart
plt.figure(figsize=(10, 6))
diab_counts.plot(kind='bar')
plt.title('Counts of Different Types of Diabetes (by phenotype)')
plt.xlabel('Phenotype')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()



In [ ]:
hip_phe = {'MS_745.11', 'MS_745.9'}

In [ ]:
DIABETES_PHECODES = set(diab_phecodes)

HIP_FRACTURE_PHECODES = hip_phe

is_diabetes = df_with_dobyr["phecode"].isin(DIABETES_PHECODES)

first_diab = (
    df_with_dobyr[is_diabetes]
    .groupby("enrolid")["first_intday"]
    .min()
    .rename("diabetes_day")
)

print(f"Patients with at least one diabetes diagnosis: {len(first_diab):,}")

# Compute first hip fracture day per patient
print("Computing earliest hip fracture day per enrolid ...")

is_hip = df_with_dobyr["phecode"].isin(HIP_FRACTURE_PHECODES)

first_hip = (
    df_with_dobyr[is_hip]
    .groupby("enrolid")["first_intday"]
    .min()
    .rename("hip_frac_day")
)

print(f"Patients with at least one hip fracture diagnosis: {len(first_hip):,}")

# Join first diabetes and hip fracture days back to a patient-level table 
# Start from the unique list of patients present in df
print("Building patient-level table with first diabetes and hip fracture days ...")

patients = df_with_dobyr[["enrolid"]].drop_duplicates().set_index("enrolid")

patients = patients.join(first_diab, how="left")
patients = patients.join(first_hip, how="left")

# Compute days between diabetes and hip fracture
# We want: hip_frac_day - diabetes_day
# - If hip fracture happens AFTER diabetes and within 365 days → label = 1
# - Otherwise (no hip fracture, or before diabetes, or >365 days) → label = 0

patients["days_to_hip_frac"] = patients["hip_frac_day"] - patients["diabetes_day"]

# Label = 1 if:
#   - both days are present (not NaN)
#   - hip fracture happens after diabetes (days_to_hip_frac > 0)
#   - and within 365 days

# cond_has_diab = patients["diabetes_day"].notna()
# cond_has_hip  = patients["hip_frac_day"].notna()
# cond_window   = (patients["days_to_hip_frac"] > 0) & (patients["days_to_hip_frac"] <= 365)

patients["hip_frac_within_1_year"] = ((patients["days_to_hip_frac"] > 0) & (patients["days_to_hip_frac"] <= 365)).astype(int)

print("\nSummary of hip_frac_within_1_year:")
print(patients["hip_frac_within_1_year"].value_counts(dropna=False))

# If you want to inspect a few examples:
print("\nExample rows:")
print(patients.reset_index().head(10).to_string(index=False))

# Optionally, save the resulting table for use with embeddings later
# patients.reset_index().to_pickle(DATA_DIR / "hip_frac_labels.pkl")


In [ ]:
print(type(patients))

In [ ]:
import pandas as pd
# def assign_class(row):                                                                                                          
#     d = row['days_to_hip_frac']                                                                                                 
#     if pd.isna(d) or d <= 0 or d > 365*5:
#         return 0          # never / too distant                                                                                 
#     elif d <= 365:
#         return 2          # within 1 year (highest risk)                                                                        
#     else:                                                                                                                       
#         return 1          # 1–5 years                                                                                           
                                                                                                                                  
patients['risk_class'] = patients.apply(assign_class, axis=1)                                                                   
# Redefine assign_class with new bins: 0=never/≤0 or >10y, 1=1–5y, 2=≤1y, 3=5–10y, 4=>10y (but last bin is 0 for legacy output as described)
def assign_class(row):
    d = row['days_to_hip_frac']
    if pd.isna(d) or d <= 0 or d > 365*10:
        return 0          # never / too distant (>10y)
    elif d <= 365:
        return 2          # within 1 year
    elif d <= 365*5:
        return 1          # 1–5 years
    elif d <= 365*10:
        return 3          # 5–10 years
    else:
        return 0          # fallback (shouldn't ever reach, but keep logic strict)

patients['risk_class'] = patients.apply(assign_class, axis=1)
print(patients['risk_class'].value_counts())

In [ ]:
patients.head()

In [ ]:
import matplotlib.pyplot as plt

counts = patients['risk_class'].value_counts().sort_index()
ax = counts.plot(kind='bar', figsize=(6, 4), color=['#6c757d', '#0d6efd', '#dc3545'])
# Make explicit labels for each bar
risk_class_labels = ['never/>10y', '1–5y', '≤1y', '5–10y']
ax.set_xticks(range(len(risk_class_labels)))
ax.set_xticklabels(risk_class_labels, rotation=0)
ax.set_title('Risk class distribution')
ax.set_xlabel('risk_class')
ax.set_ylabel('Number of patients')

# annotate bars with count + percent
n = counts.sum()
for p in ax.patches:
    h = p.get_height()
    ax.annotate(f"{int(h):,}\n({h/n:.1%})", (p.get_x() + p.get_width()/2, h),
                ha='center', va='bottom', fontsize=9, xytext=(0, 3), textcoords='offset points')

plt.tight_layout()

plt.show()

In [ ]:
patients.reset_index().to_pickle('hip_frac_labels.pkl')                                                                         
# Columns: enrolid | diabetes_day | hip_frac_day | days_to_hip_frac | hip_frac_within_1_year | risk_class    

In [ ]:
hip_phecodes = {'MS_745.11', 'MS_745.9'}
hip_phenotypes = set(
    df_with_dobyr[df_with_dobyr['phecode'].isin(hip_phecodes)]['phenotype'].unique()
)
print(hip_phenotypes)
# e.g. {"Fracture of the neck of femur", "Fracture of hip NOS"}
# Copy these exact strings into the build script below

## Different check

In [ ]:
#load embeddings
with open('data/behrt/output/patient_embeddings.pkl', 'rb') as f:
    embeddings = pickle.load(f)

embeddings.head()
print(f"Number of patients in embeddings dataframe: {embeddings.shape[0]:,}")

In [ ]:
import pickle
# checking the data that might contain gender information
with open('data/behrt/patient_info_db1.pkl', 'rb') as f:
    patient_data = pickle.load(f)

# Show the first 10 keys and their corresponding values in the patient_data dict
for i, (k, v) in enumerate(patient_data.items()):
    if i >= 10:
        break
    print(f"Key: {k}\nValue: {v}\n{'-'*50}")

